# APT Lab 01 — GRPO Ratio Clipping and Gradient Diagnostics

**Learning path:** toy GRPO clipping → mathematical diagnostic → tested observation → reuse in TRL GRPO.

**Status:** guided student lab, **not** a real GRPO trainer or an executed LLM training result. Python/PyTorch and Matplotlib on CPU are sufficient.

**Learning goals:** understand GRPO/PPO-style importance ratios, advantages, upper vs lower clipping branches, the difference between ratio clipping and gradient norm clipping, automatic differentiation, gradient cosine, and diagnostic evidence. **Four visualization experiments** below make these mechanisms inspectable.

**Workflow:** learn the motivation → derive and follow a worked example → predict → implement TODOs → compute your own plot data → interpret and test → save evidence. In Colab select **Runtime → Run all** after completing TODOs. If saving edits, use **File → Save a copy in Drive**; editing Colab does not automatically commit your GitHub repository.

## 0. Setup and reproducibility

Colab normally has PyTorch installed. This cell verifies it rather than silently upgrading the runtime.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
print('PyTorch:', torch.__version__)
print('Device: CPU (sufficient for this lab)')
torch.manual_seed(7)

## 1. Why clipping matters: learning from an older policy

Imagine the model produces several candidate code solutions, and a verifier scores each candidate. GRPO uses **group-relative rewards** to increase the likelihood of better-than-average completions and reduce the likelihood of worse-than-average ones.

Those completions came from an **old/behavior policy**. During optimization the current policy changes, but these samples are fixed. An update that excessively increases or decreases their probabilities can destabilize learning.

**Ratio clipping limits the local incentive in the training objective.** It does *not* clip model probabilities themselves or directly bound the final parameter update.

| Symbol | Meaning |
|---|---|
| $q$ | Prompt |
| $o$ | Sampled output/token (depending on objective) |
| $\pi_\theta$ | Current model |
| $\pi_{\mathrm{old}}$ | Model that generated samples |
| $r(\theta)$ | Current/old sampled-action likelihood ratio |
| $A$ | Advantage, a relative quality score |
| $\epsilon$ | Width of allowed clipping interval |

The lab isolates a **two-sample local gradient mechanism**. A real TRL GRPO run involves token-level log probabilities, multiple generated completions and additional implementation choices.

## 1.1 From verifier rewards to group-relative advantages

Consider four completions of one prompt with verifier rewards $[1,1,0,0]$. A simple advantage estimator is

$$
A_i = \frac{R_i-\bar{R}}{s_R+\delta}.
$$

Here $\bar{R}$ is the group mean, $s_R$ is its standard deviation and $\delta$ prevents division by zero. Actual GRPO implementations may vary normalization details; record the exact definition.

The successful completions have positive relative advantage; the unsuccessful ones negative advantage. **This says nothing about the reliability of the verifier**. If all rewards are tied, group-centered advantages can have zero learning signal.

**Check your understanding:** Why is advantage not interchangeable with raw reward? Why would a high rate of tied groups be a useful diagnostic metric?

## 1.2 Importance ratios and the clipping rule

For a sampled action:

$$
r(\theta)=\frac{\pi_\theta(o\mid q)}{\pi_{\mathrm{old}}(o\mid q)}
=\exp\left(\log\pi_\theta(o\mid q)-\log\pi_{\mathrm{old}}(o\mid q)\right).
$$

A ratio of $1$ indicates no relative probability change for that action; $1.3$ indicates a $30\%$ increase in likelihood relative to the old policy. It is **not** a probability of $1.3$.

The familiar maximized PPO/GRPO-style surrogate is

$$
J(\theta)=\min\left(r(\theta)A,\;\operatorname{clip}(r(\theta),1-\epsilon,1+\epsilon)A\right).
$$

For **positive advantage**, the upper clipping boundary can make the objective locally flat when $r>1+\epsilon$. For **negative advantage**, the lower boundary can make it flat when $r<1-\epsilon$.

A trainer usually *minimizes the negative* of this objective, potentially with additional terms. Distinguish the objective's gradient from **gradient norm clipping**, which is a separate optimizer operation. Do not confuse local flatness for one sample with zero total parameter gradient.

## 1.3 Fully worked numerical example — before your exercise

Suppose $\epsilon=0.2$, $A=+2$ and $r=1.1$.

1. The clipping interval is $[0.8,1.2]$.
2. Unclipped branch: $rA=1.1\times2=2.2$.
3. Clipped branch: $\operatorname{clip}(1.1,0.8,1.2)\times2=2.2$.
4. Surrogate value: $\min(2.2,2.2)=2.2$. The positive-advantage incentive remains active here.

Change only the ratio to $1.5$:

1. Unclipped branch: $1.5\times2=3.0$.
2. Clipped branch: $1.2\times2=2.4$.
3. Surrogate value: $\min(3.0,2.4)=2.4$. Pushing $r$ even higher no longer improves this sample's clipped surrogate.

**Key distinction:** The actual ratio can still be $1.5$; the loss uses the clipped branch. Nothing forces the underlying model probability to equal $1.2$.

**Pause and reason:** If $A$ were negative, which boundary would suppress the incentive to further decrease the sampled action's probability? Then attempt the next prediction exercise without looking for an answer.

## 1. Predict before coding

For the two samples below, the initial probability ratios are `1.3` and `1.0`. Both have advantage `+1`. With `epsilon = 0.2`, which sample should be suppressed by GRPO ratio clipping? Write your prediction here before running the next cell.

> **My prediction:** ...

Recall that clipping the **importance ratio in the loss** is not the same as clipping the **gradient norm**.

### Why the toy model uses log ratios

We approximate a local log-likelihood change by

$$
\log r_i(\theta)=\log r_{i,0}+x_i^\top\theta,
\qquad
r_i(\theta)=r_{i,0}\exp(x_i^\top\theta).
$$

When $\theta=0$, each ratio equals its initial value. The chain rule gives

$$
\nabla_\theta r_i(\theta)=r_i(\theta)x_i.
$$

The feature vectors therefore control how strongly each sample pulls in each parameter direction. Clipping one sample can change the *direction* of their combined gradient.

These hand-selected vectors are **not a normalized vocabulary model**. The purpose is to isolate the gradient mechanism before inspecting real TRL logs.

## 2. Build a differentiable toy ratio model

We use `log_ratio = log(initial_ratio) + feature · theta`. The feature vectors are an explicit local model of how log probabilities change with parameters. They are **not** a complete normalized token policy, so this is a controlled mechanism demonstration.

In [ ]:
theta = torch.nn.Parameter(torch.zeros(2))
features = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
initial_ratios = torch.tensor([1.3, 1.0])
advantages = torch.tensor([1.0, 1.0])
epsilon = 0.2

log_ratios = torch.log(initial_ratios) + features @ theta
ratios = torch.exp(log_ratios)
print('Ratios:', ratios.detach().tolist())

## 3. Your implementation — two lines

The positive-advantage clipped objective is `min(ratio * advantage, clamp(ratio, 1-epsilon, 1+epsilon) * advantage)`.

Replace the two `...` values. **Don't reveal the reference solution until after attempting this.**

In [ ]:
# TODO 1: multiply each probability ratio by its advantage.
unclipped = ...
# TODO 2: clamp each ratio to [1-epsilon, 1+epsilon], then multiply by its advantage.
clipped = ...

assert isinstance(unclipped, torch.Tensor), 'Implement unclipped first'
assert isinstance(clipped, torch.Tensor), 'Implement clipped first'
assert unclipped.shape == (2,) and clipped.shape == (2,)
print('Unclipped:', unclipped.detach().tolist())
print('Clipped:', clipped.detach().tolist())

## 4. Measure gradients and calculate cosine similarity

For each objective, `autograd.grad` returns the gradient with respect to `theta`. We compare the raw and clipped atomic gradients against a **fixed illustrative composition gradient** `[1, -1]`. A positive cosine means local alignment; negative means local conflict.

In [ ]:
raw_objective = unclipped.sum()
clipped_objective = torch.minimum(unclipped, clipped).sum()
g_raw = torch.autograd.grad(raw_objective, theta, retain_graph=True)[0]
g_clipped = torch.autograd.grad(clipped_objective, theta)[0]
g_composition = torch.tensor([1.0, -1.0])
raw_cos = F.cosine_similarity(g_raw[None], g_composition[None]).item()
clipped_cos = F.cosine_similarity(g_clipped[None], g_composition[None]).item()
print('Raw gradient:', g_raw.tolist())
print('Clipped gradient:', g_clipped.tolist())
print('Raw cosine:', round(raw_cos, 3))
print('Clipped cosine:', round(clipped_cos, 3))

## 5. Diagnostic plot 1 — where does the GRPO objective become flat?

**Your assignment:** plot the *unclipped* and *clipped* surrogate against the probability ratio for both **positive and negative advantages**. Predict which side flattens before running it.

**You implement:** the mathematical curves (the `TODO`s below). The plotting scaffold handles consistent layout, labels, annotation, legend, and export. Do not treat plotting as decoration: write down which mechanism each observed shape demonstrates.

**Acceptance:** both advantage signs, clip boundaries, axes with units/meaning, and a written description of the plateau. No production trainer claims from this toy demonstration.

In [ ]:
from pathlib import Path
PLOT_DIR = Path("figures")
PLOT_DIR.mkdir(exist_ok=True)

# Reusable publication-ready plotting defaults; change style if useful.
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 220,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.2,
    "font.size": 11, "axes.titlesize": 12,
})

ratio_grid = np.linspace(0.4, 1.6, 401)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)

for ax, advantage in zip(axes, (+1.0, -1.0)):
    # TODO PLOT 1A: calculate raw_values = rho * A as a vector.
    raw_values = None
    # TODO PLOT 1B: calculate clipped_values = min(raw, clipped_ratio * A).
    clipped_values = None

    if raw_values is None or clipped_values is None:
        raise NotImplementedError("Implement both surrogate curves first")

    ax.plot(ratio_grid, raw_values, "--", lw=2, label="Unclipped surrogate")
    ax.plot(ratio_grid, clipped_values, lw=2.5, label="Clipped surrogate")
    for edge in (1 - epsilon, 1 + epsilon):
        ax.axvline(edge, ls=":", lw=1.4, color="0.4")
    ax.set(title=f"Advantage = {advantage:+g}",
           xlabel="Importance ratio ρ", ylabel="Surrogate value")
    ax.legend(loc="best", frameon=False)
fig.suptitle("Diagnostic 1 — Where does ratio clipping suppress improvement?")
fig.tight_layout()
fig.savefig(PLOT_DIR / "01_surrogate_shapes.png", bbox_inches="tight")
plt.show()

## 6. Diagnostic plot 2 — which gradients are actually suppressed?

Plot the derivative $dJ/d\rho$ against the importance ratio, separately for positive and negative advantages.

**You implement:** compute the clipped surrogate using PyTorch and call `torch.autograd.grad`. **Do not** plot a hand-written zero/one step function; use autograd to validate the actual objective. Then explain how $dJ/d\rho$ differs from $\nabla_\theta J$ by the chain rule.

**Acceptance:** identify the zero-gradient region for each sign and explain what is undefined/ambiguous at exact boundaries.

In [ ]:
ratio_samples = torch.tensor(
    np.linspace(0.405, 1.595, 240),
    dtype=torch.float64, requires_grad=True
)
fig, ax = plt.subplots(figsize=(10, 4.5))

for advantage in (+1.0, -1.0):
    # TODO PLOT 2A: construct unclipped and clipped ratio surrogate
    # using ratio_samples and this advantage.
    surrogate = None
    if surrogate is None:
        raise NotImplementedError("Implement the clipped surrogate on ratio_samples")

    # TODO PLOT 2B: use torch.autograd.grad to calculate d(sum(surrogate))/d(ratio_samples)
    derivative = None
    if derivative is None:
        raise NotImplementedError("Compute the derivative with autograd")

    ax.plot(ratio_samples.detach().numpy(),
            derivative.detach().numpy(), lw=2.3,
            label=f"Advantage = {advantage:+g}")

for edge in (1 - epsilon, 1 + epsilon):
    ax.axvline(edge, ls=":", lw=1.5, color="0.4")
ax.set(title="Diagnostic 2 — Derivative of the clipped surrogate",
       xlabel="Importance ratio ρ", ylabel="dJ / dρ")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOT_DIR / "02_ratio_derivative.png", bbox_inches="tight")
plt.show()

## 7. Diagnostic plot 3 — why can cosine similarity change sign?

**Your assignment:** produce a vector plot showing **raw atomic**, **clipped atomic**, and **composition** gradients on the same axes. Label all three, preserve equal axis aspect ratio, and annotate the *two* cosine similarities already measured.

**You implement:** assemble the vectors to be plotted. The rendering helper is provided so you can concentrate on gradient reasoning.

**Acceptance:** explain how removing one sample's gradient changes the *direction* of the aggregate gradient, not merely its magnitude.

In [ ]:
def draw_gradient_vectors(vectors, *, output_path, title):
    """Render named two-dimensional vectors with readable labels and equal scale."""
    fig, ax = plt.subplots(figsize=(7.5, 6))
    for label, vector in vectors.items():
        arr = np.asarray(vector, dtype=float)
        if arr.shape != (2,):
            raise ValueError(f"{label}: expected exactly two coordinates")
        ax.quiver(0, 0, arr[0], arr[1], angles="xy", scale_units="xy",
                  scale=1, width=0.008, label=label)
    ax.axhline(0, color="0.35", lw=0.8)
    ax.axvline(0, color="0.35", lw=0.8)
    max_extent = max(1.0, *(np.max(np.abs(v)) for v in vectors.values())) * 1.25
    ax.set(xlim=(-max_extent, max_extent), ylim=(-max_extent, max_extent),
           xlabel="Parameter direction θ₁", ylabel="Parameter direction θ₂",
           title=title)
    ax.set_aspect("equal")
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()

# TODO PLOT 3: provide a dictionary of three labeled 2D gradient vectors,
# taking values from g_raw, g_clipped, and g_composition.
gradient_vectors = None
if gradient_vectors is None:
    raise NotImplementedError("Assemble all three gradient vectors")

draw_gradient_vectors(
    gradient_vectors,
    output_path=PLOT_DIR / "03_gradient_alignment.png",
    title=f"Raw cosine {raw_cos:+.3f} vs clipped cosine {clipped_cos:+.3f}",
)

## 8. Diagnostic plot 4 — choose an intervention and measure sensitivity

The first sample has $\rho=1.3$ and positive advantage. Vary clipping threshold $\epsilon$ while holding every other value constant.

**Your assignment:** measure (i) that sample's actual parameter-gradient contribution and (ii) the full batch's cosine against the composition gradient at each threshold. Plot both against $\epsilon$.

**You implement:** the experimental sweep and metric calculation. The scaffold provides two aligned publication-quality panels, labels, and saved image.

**Predict:** at which threshold will the first sample stop being suppressed? Avoid the exact equality in your test values because automatic differentiation at a kink can be implementation-specific.

**Acceptance:** a justified threshold, a correct plot, and a statement of what this toy result does *not* prove about held-out accuracy.

In [ ]:
eps_values = [0.05, 0.10, 0.20, 0.25, 0.31, 0.40]
sample0_gradient_norms = []
batch_cosines = []

for eps in eps_values:
    t = torch.zeros(2, requires_grad=True)
    current_ratios = torch.exp(torch.log(initial_ratios) + features @ t)
    # TODO PLOT 4A: implement per-sample clipped surrogate at the current eps
    per_sample_surrogate = None
    if per_sample_surrogate is None:
        raise NotImplementedError("Implement per-sample clipped surrogate")

    # TODO PLOT 4B: compute sample-0 and aggregate parameter gradients using autograd
    sample0_grad = None
    batch_grad = None
    if sample0_grad is None or batch_grad is None:
        raise NotImplementedError("Compute both parameter gradients")

    # TODO PLOT 4C: append norm(sample0_grad) and cosine(batch_grad, g_composition)
    raise NotImplementedError("Append both diagnostics for this epsilon")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharex=True)
axes[0].plot(eps_values, sample0_gradient_norms, "o-", lw=2.2)
axes[0].set(xlabel="Clipping threshold ε", ylabel="Sample 0 gradient norm",
            title="Does sample 0 contribute?")
axes[1].plot(eps_values, batch_cosines, "o-", lw=2.2)
axes[1].axhline(0, ls=":", color="0.4")
axes[1].set(xlabel="Clipping threshold ε", ylabel="Cosine with composition gradient",
            title="Does effective gradient direction change?")
fig.suptitle("Diagnostic 4 — Sensitivity to a controlled clipping intervention")
fig.tight_layout()
fig.savefig(PLOT_DIR / "04_clip_sensitivity.png", bbox_inches="tight")
plt.show()

## 9. Your diagnostic interpretation (required before proceeding)

Write a short research note under this cell answering:

1. **Plot 1:** Which side of the ratio interval is flat for each advantage sign? Why?
2. **Plot 2:** How does the derivative confirm your explanation? Distinguish $dJ/d\rho$ from $\nabla_\theta J$.
3. **Plot 3:** Why did the cosine change sign? Which sample's contribution disappeared?
4. **Plot 4:** Which controlled intervention changed the sign of the gradient alignment? What mechanism was held constant?
5. **Research caution:** Does this establish that clipping caused held-out composition regression in a real GRPO run? What additional experiment is needed?

**Finish criteria:** four saved figures, three TODOs in the preceding autograd exercise, test assertions, and an evidence report with the correct scope. Next trainer lab: TRL SFTTrainer with loss masks and gradient diagnostics.

## 10. Diagnostic tests — student exercise

Add an assertion that ensures (a) raw cosine is positive, (b) clipped cosine is negative. Then test the gradient vectors with `torch.testing.assert_close`. The reference expectations for this toy setup are `g_raw=[2.6,1]` and `g_clipped=[0,1]`.

In [ ]:
# TODO 3: replace `pass` with your assertions.
# Helpful examples: assert raw_cos > 0 ; torch.testing.assert_close(g_raw, torch.tensor([...]))
pass

## 11. Turn observations into a simple APT report

This reports a verified **toy clipping sign-flip mechanism**, not a claim that clipping caused a production held-out regression.

In [ ]:
report = {
    'experiment_id': 'toy_grpo_clipping_01',
    'evidence_scope': 'controlled_toy_surrogate',
    'ratios': [float(v) for v in ratios.detach()],
    'clip_epsilon': epsilon,
    'raw_gradient_cosine': raw_cos,
    'clipped_gradient_cosine': clipped_cos,
    'sign_flip': bool(raw_cos > 0 and clipped_cos < 0),
    'diagnosis': 'clipping_changes_gradient_direction_in_this_toy_example',
}
import json
from pathlib import Path
Path('apt_lab01_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 12. Investigate — don't just copy outputs

1. Change the first `initial_ratios` value from `1.3` to `1.1`. What happens to the clipped gradient?
2. Change its advantage to `-1` and reason about which side of the clipping interval suppresses negative-advantage gradients.
3. Explain why a sign flip **does not prove** that clipping caused a real held-out accuracy regression.

**Submit to your tutor:** your two TODO lines, the assertion test, outputs, and brief explanation. The next notebook will preserve the report shape while moving to a real TRL `SFTTrainer` experiment with label-mask diagnostics.

## Reference solution (open only after attempting)

For TODO 1–2:

```python
unclipped = ratios * advantages
clipped = torch.clamp(ratios, 1 - epsilon, 1 + epsilon) * advantages
```

For TODO 3:

```python
assert raw_cos > 0
assert clipped_cos < 0
torch.testing.assert_close(g_raw, torch.tensor([2.6, 1.0]), atol=1e-5, rtol=1e-5)
torch.testing.assert_close(g_clipped, torch.tensor([0.0, 1.0]))
```

Run outputs will be generated only when you execute the notebook.